# CD10-1 PostgreSQL Architecture · Colab 실습
왼쪽 열쇠(보안 비밀)에 `PG_HOST`, `PG_USER`, `PG_PASSWORD`를 등록하고, RDS 보안그룹에서 5432 포트를 허용한 뒤 실행하세요.

In [ ]:
!pip install -q psycopg2-binary

## 실습 1 · 연결 2개 = 백엔드 프로세스 2개 (Client / Server Process)

In [ ]:
from google.colab import userdata
import psycopg2
cfg = dict(host=userdata.get('PG_HOST').strip(), port=5432, dbname='dvdrental',
           user=userdata.get('PG_USER').strip(),
           password=userdata.get('PG_PASSWORD').strip())
c1, c2 = psycopg2.connect(**cfg), psycopg2.connect(**cfg)
for c in (c1, c2):
    cur = c.cursor(); cur.execute('SELECT pg_backend_pid()')
    print(cur.fetchone())

## 실습 2 · 메모리 설정과 Utility 프로세스 관찰 (System Memory / Utility)

In [ ]:
import pandas as pd
cur = c1.cursor()
for p in ['shared_buffers', 'work_mem', 'wal_buffers', 'max_connections']:
    cur.execute(f'SHOW {p}'); print(p, '=', cur.fetchone()[0])

q = '''SELECT backend_type, count(*) AS n
       FROM pg_stat_activity GROUP BY 1 ORDER BY 2 DESC'''
cur.execute(q)
display(pd.DataFrame(cur.fetchall(), columns=['backend_type', 'n']))

## 실습 3 · SQL 5단계를 오류와 실행계획으로 관찰 (Parser → Executor)

In [ ]:
c1.autocommit = True; cur = c1.cursor()
for sql in ['SELEC * FROM film', 'SELECT * FROM filmx']:
    try: cur.execute(sql)
    except Exception as e: print(type(e).__name__, '→', str(e).splitlines()[0])

q = 'SELECT sum(amount) FROM payment WHERE amount >= 3'
for i in (1, 2):
    cur.execute('EXPLAIN (ANALYZE, BUFFERS) ' + q)
    print(f'--- {i}차'); print('\n'.join(r[0] for r in cur.fetchall()))

## 심화 · work_mem과 Sort Method (Local Memory)

In [ ]:
for wm in ('4MB', '64kB'):
    cur.execute(f"SET work_mem = '{wm}'")
    cur.execute('EXPLAIN ANALYZE SELECT * FROM payment ORDER BY amount, payment_date')
    print(wm, [r[0].strip() for r in cur.fetchall() if 'Sort Method' in r[0]])
cur.execute('RESET work_mem')

In [ ]:
c1.close(); c2.close()